In [1]:
# Import required libraries and initialize the Neo4j database connection

from neo4j import GraphDatabase
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password123")

driver = GraphDatabase.driver(URI, auth=AUTH)

In [2]:
# Query graph data for analytics

with driver.session() as session:
    # Distribution query
    dfEngagement = pd.DataFrame(session.run("""
        MATCH (q:Question)
        RETURN q.view_count AS Views, q.answer_count AS Answers, q.title AS Title
    """).data())
    
    # Top answers score evaluation
    dfScores = pd.DataFrame(session.run("""
        MATCH (a:Answer)
        RETURN a.score AS Score, a.is_accepted AS IsAccepted
    """).data())

driver.close()

In [3]:
# Display Engagement Scatter Plot
figScatter = px.scatter(
    dfEngagement, 
    x="Views", 
    y="Answers", 
    hover_name="Title",
    title="Question Engagement Analysis: Views vs. Answers",
    labels={"Views": "View Count", "Answers": "Answer Count"},
    template="plotly_dark",
    color="Answers",
    color_continuous_scale="Viridis"
)
figScatter.show()

In [4]:
# Display answer score distribution
figBox = px.box(
    dfScores, 
    x="IsAccepted", 
    y="Score", 
    title="Score Distribution by Answer Acceptance Status",
    labels={"IsAccepted": "Is Accepted Answer", "Score": "Score"},
    template="plotly_dark",
    color="IsAccepted"
)
figBox.show()